# 02 — Tokenizer BPE

Projet GPT reproductible · Notebook en français.


## 1. Research objective

Question : comment la taille de vocabulaire affecte-t-elle la fragmentation ? Hypothèse : vocabulaire plus grand, séquence souvent plus courte, mais empreinte accrue.


## 2. Configuration

Graine 2026; dépendance tiktoken. API `_educational` pédagogique, non garantie stable en production.


## 3. Implementation

BPETokenizer fournit train/encode/decode/save/load. Rangs d’octets sauvegardés en base64 JSON.


In [1]:
import base64,json,random
from pathlib import Path
from tiktoken._educational import SimpleBytePairEncoding, bpe_train
SEED=2026; random.seed(SEED)
PAT=r"'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"
TEXT=("A language model predicts tokens. Byte pair encoding merges bytes. ")*30
class BPETokenizer:
 def __init__(self,enc,size,seed): self.enc,self.size,self.seed=enc,size,seed
 @classmethod
 def train(cls,text,vocab_size=300,seed=SEED):
  if not text.strip() or vocab_size<256: raise ValueError('Corpus vide ou vocabulaire inférieur à 256')
  return cls(SimpleBytePairEncoding(pat_str=PAT,mergeable_ranks=bpe_train(data=text,vocab_size=vocab_size,pat_str=PAT,visualise=None)),vocab_size,seed)
 def encode(self,text): return self.enc.encode(text,visualise=None)
 def decode(self,ids): return self.enc.decode(ids)
 def save(self,path):
  d={'size':self.size,'seed':self.seed,'pat':self.enc.pat_str,'ranks':[[base64.b64encode(k).decode(),v] for k,v in self.enc.mergeable_ranks.items()]}
  Path(path).write_text(json.dumps(d),encoding='utf-8')
 @classmethod
 def load(cls,path):
  d=json.loads(Path(path).read_text(encoding='utf-8')); ranks={base64.b64decode(k):v for k,v in d['ranks']}
  return cls(SimpleBytePairEncoding(pat_str=d['pat'],mergeable_ranks=ranks),d['size'],d['seed'])
OUT=Path('../artifacts/tokenizers'); OUT.mkdir(parents=True,exist_ok=True)
tok=BPETokenizer.train(TEXT,300)


## 4. Unit tests

Round-trip Unicode et IDs identiques après rechargement.


In [2]:
texts=['Hello, world!','café — 東京','Attention causale']
for t in texts: assert tok.decode(tok.encode(t))==t
path=OUT/'bpe_demo.json'; tok.save(path); clone=BPETokenizer.load(path)
for t in texts: assert clone.encode(t)==tok.encode(t)
print('Tests round-trip/save/load: OK')


Tests round-trip/save/load: OK


## 5. Experiments

Comparer différentes tailles sur un texte de validation fixe.


In [3]:
validation=('A language model predicts tokens. ')*8
rows=[]
for size in (260,280,300):
 model=BPETokenizer.train(TEXT,size); n=len(model.encode(validation))
 rows.append({'vocab':size,'tokens':n,'chars/token':round(len(validation)/n,2)})
rows


[{'vocab': 260, 'tokens': 240, 'chars/token': 1.13},
 {'vocab': 280, 'tokens': 96, 'chars/token': 2.83},
 {'vocab': 300, 'tokens': 56, 'chars/token': 4.86}]

## 6. Results

Nombre de tokens et caractères/token.


In [4]:
rows


[{'vocab': 260, 'tokens': 240, 'chars/token': 1.13},
 {'vocab': 280, 'tokens': 96, 'chars/token': 2.83},
 {'vocab': 300, 'tokens': 56, 'chars/token': 4.86}]

## 7. Discussion

Résultats propres au corpus. Figer la version de tiktoken et mesurer plusieurs langues avant toute conclusion.
